<h1>🩺 Biofilter — Report: <code>annotation_master_disease</code></h1>

Everything the bundle knows about a list of diseases: MONDO record, groups, cross-references grouped by the source that issued them, and how many genes ClinGen links to the disease.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# A bundle is a directory — the one holding manifest.json.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotation_master_disease"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Diseases resolve by MONDO id, by label, or by a cross-reference code from
any source the bundle carries.

In [ ]:
diseases = [
    "MONDO:0007254",     # breast cancer, by id
    "Leigh syndrome",    # by label
    "NOT_A_DISEASE",     # kept, with status='not_found'
]

result = bf.report.run(REPORT, input_data=diseases)
df = result.to_pandas()
df[["input_value", "disease_id", "disease_label", "omic_status", "status"]]

### 4. The two ClinGen numbers

`clingen_gene_count` counts **distinct genes**; `clingen_relationship_count`
counts assertions. One gene supported by three lines of evidence is one
gene and three assertions — when they diverge, that is why.

In [ ]:
df[[
    "input_value",
    "clingen_gene_count",
    "clingen_relationship_count",
    "total_entity_relationships",
]]

ClinGen's share is not the total. A disease can have thousands of
relationships — MONDO's own hierarchy, Reactome, BioGRID — and no ClinGen
genes at all. That means nobody has curated a gene–disease assertion for
it, not that it has no genetic basis.

### 5. Cross-references, grouped by who issued them

In [ ]:
for _, row in df[df["status"] == "ok"].iterrows():
    print(row["disease_label"])
    for entry in row["xref_ids_by_source"]:
        print(f"  {entry['source']:<12} {list(entry['ids'])[:4]}")
    print("  groups:", list(row["disease_groups"]))
    print()

### 6. Every disease in the bundle

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__", include_clingen_summary=False)
catalog = everything.to_pandas()

print(f"{everything.num_rows:,} diseases in {time.perf_counter() - started:.1f}s")
catalog["status"].value_counts()

### 7. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the ids came from.

In [ ]:
for path in result.write("annotation_master_disease.csv"):
    print(path)

### 8. The same thing on the command line

```bash
biofilter report run --report-name annotation_master_disease \\
    --input ... \\
    --output out.csv
```

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))